# Bài 2.7 · Gradient boosting trên dữ liệu thật

**Câu hỏi của buổi này:** trên một bảng dữ liệu thật, gradient boosting hơn baseline bao nhiêu, dừng ở cây thứ mấy, và dựa vào cột nào?

Bộ dữ liệu **Adult** (UCI, điều tra dân số Mỹ 1994) có gần 49 000 người, mỗi người 14 cột: tuổi, học vấn, nghề, số giờ làm mỗi tuần… Nhiệm vụ là đoán thu nhập năm có vượt 50 000 USD không. Metric là **ROC-AUC**, vì lớp dương chỉ chiếm khoảng một phần tư.

| Mục | Sẽ thấy |
|---|---|
| 1. Dữ liệu | bảng mẫu, tỉ lệ hai lớp |
| 2. Khuôn đánh giá | bảng chia test và 5 fold |
| 3. Baseline | AUC của hằng số và của logistic regression |
| 4. LightGBM có early stopping | AUC từng fold, đường AUC theo số cây, điểm dừng |
| 5. Learning rate và số cây | ba đường AUC, bảng số cây và thời gian |
| 6. Mô hình dựa vào cột nào | độ quan trọng theo gain; bỏ cột mạnh nhất thì mất bao nhiêu |
| 7. XGBoost, CatBoost, trung bình | bảng và biểu đồ năm cách |
| 8. Tóm tắt | một bảng các con số chính |

Các dòng ✓ là phép kiểm in ra: chạy lại mà thấy ✗ thì có gì đó sai. Bật **Internet** trong *Session options*; CPU là đủ, chạy hết khoảng 4 phút. Bài giảng: [ai-canuc80k · bài 2.7](https://ai-canuc80k.ai-web-deploy-20261010.workers.dev/#/lesson/2.7).

In [ ]:
# Ô tiện ích: định dạng bảng, dòng kiểm ✓/✗, nhãn số trên biểu đồ. Không cần đọc để theo bài.
import warnings, contextlib, io, logging
warnings.filterwarnings('ignore')
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from IPython.display import display, HTML

plt.rcParams.update({
    'figure.dpi': 110, 'figure.figsize': (8, 4), 'font.size': 11, 'axes.titlesize': 12, 'axes.titleweight': 'bold',
    'axes.spines.top': False, 'axes.spines.right': False, 'axes.grid': True, 'grid.alpha': .25,
    'legend.frameon': False, 'image.cmap': 'viridis',
})
C = {'blue': '#3b6ea8', 'clay': '#c2603e', 'sand': '#b8892b', 'green': '#4d7337', 'gray': '#8a877f'}
ORDER = [C['blue'], C['clay'], C['sand'], C['green'], C['gray']]
plt.rcParams['axes.prop_cycle'] = plt.cycler(color=ORDER)


def table(df, title=None, fmt=None, best=None, lower_is_better=False, index=False):
    """In một bảng gọn: định dạng số, tô đậm ô tốt nhất của cột `best`."""
    st = df.style.format(fmt, na_rep='–')
    if best is not None:
        col = df[best]
        target = col.min() if lower_is_better else col.max()
        st = st.apply(lambda s: ['font-weight:700;background:#f5e6dd' if (s.name == best and v == target) else '' for v in s], axis=0)
    text_cols = [c for c in df.columns if not pd.api.types.is_numeric_dtype(df[c])]
    if text_cols:
        st = st.set_properties(subset=text_cols, **{'text-align': 'left'})
    if not index:
        st = st.hide(axis='index')
    st = st.set_table_styles([
        {'selector': 'th', 'props': 'text-align:left;padding:6px 12px;border-bottom:2px solid #d3cebf;font-weight:600'},
        {'selector': 'td', 'props': 'padding:5px 12px;border-bottom:1px solid #e6e2d6'},
        {'selector': 'caption', 'props': 'caption-side:top;text-align:left;font-weight:700;font-size:14px;padding:4px 0 8px;white-space:nowrap'},
    ])
    if title:
        st = st.set_caption(title)
    display(st)


def check(msg, ok):
    """Dòng kiểm tra in ra: ✓ khi đúng, ✗ khi sai (không dừng notebook)."""
    print(('✓ ' if ok else '✗ ') + msg)


def bar_labels(ax, fmt='{:.2f}'):
    """Ghi giá trị lên đầu mỗi thanh của biểu đồ cột."""
    for cont in ax.containers:
        ax.bar_label(cont, labels=[fmt.format(v) for v in cont.datavalues], padding=3, fontsize=9)


@contextlib.contextmanager
def quiet():
    """Ẩn thanh tiến trình và dòng thông báo khi tải dữ liệu hay trọng số."""
    logging.disable(logging.WARNING)
    try:
        with contextlib.redirect_stdout(io.StringIO()), contextlib.redirect_stderr(io.StringIO()):
            yield
    finally:
        logging.disable(logging.NOTSET)


def note(text):
    """Một dòng nhận xét in đậm nhẹ dưới output, tính từ số thật của lần chạy."""
    display(HTML(f'<div style="border-left:3px solid #c2603e;padding:4px 10px;margin:6px 0;color:#444">{text}</div>'))

In [ ]:
import time
import lightgbm as lgb, xgboost as xgb, catboost, sklearn
from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.metrics import roc_auc_score
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LogisticRegression
SEED = 0
table(pd.DataFrame([('lightgbm', lgb.__version__), ('xgboost', xgb.__version__), ('catboost', catboost.__version__), ('scikit-learn', sklearn.__version__)],
                   columns=['thành phần', 'phiên bản']), 'Môi trường chạy')

## 1. Dữ liệu

`fetch_openml` tải bảng từ OpenML. Cột chữ được giữ kiểu `category` để LightGBM, XGBoost và CatBoost chia trực tiếp theo nhóm, không cần one-hot.

In [ ]:
adult = fetch_openml('adult', version=2, as_frame=True)
X = adult.data.copy()
y = (adult.target == '>50K').astype(int).to_numpy()
cat_cols = X.select_dtypes(['category', 'object']).columns.tolist()
num_cols = [c for c in X.columns if c not in cat_cols]
for c in cat_cols:
    X[c] = X[c].astype('category')
table(X.head(5), 'Năm dòng đầu của Adult')
check(f'{len(X)} người, {len(num_cols)} cột số, {len(cat_cols)} cột chữ', len(X) == 48842)

fig, ax = plt.subplots(figsize=(8, 2))
ax.barh(['≤ 50K', '> 50K'], np.bincount(y), color=[C['gray'], C['blue']]); bar_labels(ax, '{:.0f}')
ax.set_title(f'Lớp dương (> 50K) chiếm {y.mean():.1%}'); ax.grid(axis='y', visible=False)
plt.tight_layout(); plt.show()

## 2. Khuôn đánh giá

20% dữ liệu để riêng làm **test**, chỉ chấm một lần ở cuối. Mọi lựa chọn (mô hình, learning rate, số cây) dựa trên điểm **out-of-fold (OOF)**: chia phần train thành 5 fold, mỗi fold lần lượt làm validation, ghép dự đoán của 5 lần lại thành một vector phủ đủ phần train.

In [ ]:
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, stratify=y, random_state=SEED)
folds = list(StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED).split(X_tr, y_tr))
table(pd.DataFrame([('train (5 fold)', len(y_tr), y_tr.mean()), ('test', len(y_te), y_te.mean())] +
                   [(f'  fold {k + 1} làm validation', len(b), y_tr[b].mean()) for k, (a, b) in enumerate(folds)],
                   columns=['phần', 'số người', 'tỉ lệ > 50K']), 'Chia dữ liệu', fmt={'tỉ lệ > 50K': '{:.3f}'})
check('5 fold validation phủ đúng mỗi người một lần', sorted(np.concatenate([b for _, b in folds]).tolist()) == list(range(len(y_tr))))

## 3. Baseline

Đoán cùng một xác suất cho mọi người cho AUC đúng 0.5: mô hình nào cũng phải vượt mốc này. Baseline mạnh hơn là logistic regression trên one-hot của cột chữ và cột số đã chuẩn hóa.

In [ ]:
auc_const = roc_auc_score(y_te, np.full(len(y_te), y_tr.mean()))
pre = ColumnTransformer([('cat', OneHotEncoder(handle_unknown='ignore'), cat_cols), ('num', StandardScaler(), num_cols)])
logit = make_pipeline(pre, LogisticRegression(max_iter=2000))
X_tr_lr = X_tr.copy(); X_tr_lr[cat_cols] = X_tr_lr[cat_cols].astype(str)
auc_lr = cross_val_score(logit, X_tr_lr, y_tr, cv=folds, scoring='roc_auc')
table(pd.DataFrame([('hằng số', auc_const, np.nan), ('logistic regression', auc_lr.mean(), auc_lr.std())], columns=['baseline', 'AUC', 'lệch chuẩn 5 fold']),
      'Hai baseline', fmt={'AUC': '{:.4f}', 'lệch chuẩn 5 fold': '{:.4f}'})
check('hằng số cho AUC 0.5', abs(auc_const - 0.5) < 1e-9)

## 4. LightGBM trong 5 fold, có early stopping

Mỗi fold học tối đa 5 000 cây, sau mỗi cây đo AUC trên fold validation, **dừng** khi 100 cây liên tiếp không cải thiện, rồi giữ lại số cây tốt nhất. Dự đoán test là trung bình của 5 mô hình.

In [ ]:
def run_lgb(params, rounds=5000):
    oof = np.zeros(len(y_tr)); pred = np.zeros(len(y_te)); iters, curves = [], []
    for a, b in folds:
        m = lgb.LGBMClassifier(n_estimators=rounds, random_state=SEED, verbose=-1, **params)
        m.fit(X_tr.iloc[a], y_tr[a], eval_set=[(X_tr.iloc[b], y_tr[b])], eval_metric='auc',
              callbacks=[lgb.early_stopping(100, verbose=False), lgb.record_evaluation(rec := {})])
        oof[b] = m.predict_proba(X_tr.iloc[b])[:, 1]
        pred += m.predict_proba(X_te)[:, 1] / len(folds)
        iters.append(m.best_iteration_); curves.append(np.array(rec['valid_0']['auc']))
    return oof, pred, iters, curves, m

BASE = dict(num_leaves=31, subsample=0.8, subsample_freq=1, colsample_bytree=0.8)
t = time.time()
oof_lgb, pred_lgb, it_lgb, cur_lgb, last_lgb = run_lgb(dict(learning_rate=0.05, **BASE))
t_lgb = time.time() - t
per_fold = pd.DataFrame({'fold': range(1, 6), 'dừng ở cây': it_lgb, 'AUC fold validation': [roc_auc_score(y_tr[b], oof_lgb[b]) for _, b in folds]})
table(per_fold, f'LightGBM, learning rate 0.05 ({t_lgb:.0f} giây)', fmt={'AUC fold validation': '{:.4f}'})

fig, ax = plt.subplots(figsize=(10, 4))
for k, (cv, it) in enumerate(zip(cur_lgb, it_lgb)):
    ax.plot(np.arange(1, len(cv) + 1), cv, lw=1.3, color=ORDER[k % 5], label=f'fold {k + 1}')
    ax.scatter([it], [cv[it - 1]], color=ORDER[k % 5], zorder=3, s=30)
ax.set(xlabel='số cây', ylabel='AUC trên fold validation', title='AUC tăng nhanh rồi đi ngang; chấm là cây tốt nhất, đường kéo thêm 100 cây rồi dừng')
ax.set_ylim(min(c[20] for c in cur_lgb) - 0.005, max(c.max() for c in cur_lgb) + 0.003); ax.legend(ncol=5, loc='lower right')
plt.tight_layout(); plt.show()
auc_oof_lgb, auc_te_lgb = roc_auc_score(y_tr, oof_lgb), roc_auc_score(y_te, pred_lgb)
check('LightGBM vượt logistic trên OOF', auc_oof_lgb > auc_lr.mean())
note(f'OOF AUC <b>{auc_oof_lgb:.4f}</b>, test AUC <b>{auc_te_lgb:.4f}</b>: hai số gần nhau, khuôn 5 fold ước lượng đúng điểm trên dữ liệu chưa thấy. '
     f'Hơn logistic {auc_oof_lgb - auc_lr.mean():+.4f}. Các fold dừng từ cây {min(it_lgb)} đến cây {max(it_lgb)}.')

## 5. Learning rate và số cây

Learning rate nhỏ thì mỗi cây chỉ sửa một phần nhỏ phần dư, nên cần nhiều cây hơn.

In [ ]:
rows, lr_curves = [], {}
for lr in [0.3, 0.1, 0.03]:
    t = time.time()
    oof, _, iters, curves, _ = run_lgb(dict(learning_rate=lr, **BASE))
    rows.append((lr, int(np.mean(iters)), roc_auc_score(y_tr, oof), time.time() - t)); lr_curves[lr] = curves[0]
lr_df = pd.DataFrame(rows, columns=['learning rate', 'số cây (trung bình 5 fold)', 'OOF AUC', 'giây'])
table(lr_df, 'Ba learning rate', fmt={'learning rate': '{:g}', 'OOF AUC': '{:.4f}', 'giây': '{:.1f}'}, best='OOF AUC')

fig, ax = plt.subplots(figsize=(10, 3.8))
for (lr, cv), c in zip(lr_curves.items(), [C['clay'], C['sand'], C['blue']]):
    ax.plot(np.arange(1, len(cv) + 1), cv, color=c, label=f'learning rate {lr}')
ax.set_xscale('log'); ax.set(xlabel='số cây (thang log)', ylabel='AUC fold 1', title='Learning rate nhỏ: lên chậm hơn, cần nhiều cây hơn, đỉnh nhỉnh hơn một chút')
ax.set_ylim(0.88, max(c.max() for c in lr_curves.values()) + 0.003); ax.legend()
plt.tight_layout(); plt.show()
note(f"Learning rate 0.3 dừng ở khoảng <b>{lr_df.iloc[0, 1]}</b> cây, 0.03 ở khoảng <b>{lr_df.iloc[2, 1]}</b> cây, "
     f"trong khi OOF AUC chỉ từ {lr_df.iloc[0, 2]:.4f} lên {lr_df.iloc[2, 2]:.4f}. Trong phòng thi, chọn learning rate theo thời gian còn lại.")

## 6. Mô hình dựa vào cột nào

Độ quan trọng theo **gain** là tổng độ giảm loss mà các phép chia trên một cột mang lại. Nó cho biết mô hình dùng cột nào nhiều, không chứng minh quan hệ nhân quả. Kiểm bằng cách **bỏ hẳn** cột mạnh nhất rồi chạy lại.

In [ ]:
imp = pd.Series(last_lgb.booster_.feature_importance('gain'), index=X_tr.columns).sort_values()
imp = imp / imp.sum()
fig, ax = plt.subplots(figsize=(9, 4.4))
ax.barh(imp.index, imp.values, color=[C['clay'] if v == imp.max() else C['blue'] for v in imp.values]); bar_labels(ax, '{:.1%}')
ax.set(xlabel='tỉ lệ gain', title='LightGBM: độ quan trọng theo gain (fold cuối)'); ax.grid(axis='y', visible=False)
plt.tight_layout(); plt.show()

top = imp.index[-1]; second = imp.index[-2]
abl = []
for drop in [top, second]:
    keep = [c for c in X_tr.columns if c != drop]
    oof = np.zeros(len(y_tr))
    for a, b in folds:
        m = lgb.LGBMClassifier(n_estimators=5000, learning_rate=0.05, random_state=SEED, verbose=-1, **BASE)
        m.fit(X_tr.iloc[a][keep], y_tr[a], eval_set=[(X_tr.iloc[b][keep], y_tr[b])], eval_metric='auc', callbacks=[lgb.early_stopping(100, verbose=False)])
        oof[b] = m.predict_proba(X_tr.iloc[b][keep])[:, 1]
    abl.append((f'bỏ cột {drop}', roc_auc_score(y_tr, oof)))
ab = pd.DataFrame([('đủ 14 cột', auc_oof_lgb)] + abl, columns=['thử', 'OOF AUC'])
ab['đổi'] = ab['OOF AUC'] - auc_oof_lgb
table(ab, 'Bỏ một cột rồi chạy lại 5 fold', fmt={'OOF AUC': '{:.4f}', 'đổi': '{:+.4f}'})
note(f"Bỏ <b>{top}</b> (gain {imp[top]:.0%}) làm AUC đổi {ab['đổi'].iloc[1]:+.4f}; bỏ <b>{second}</b> đổi {ab['đổi'].iloc[2]:+.4f}. "
     f"Gain lớn chưa chắc là mất nhiều nhất khi bỏ: cột khác có thể gánh thay phần thông tin trùng.")

## 7. XGBoost, CatBoost và trung bình ba mô hình

Hai thư viện còn lại chạy trong cùng khuôn fold và cùng early stopping. Ba mô hình sai theo những cách hơi khác nhau, nên trung bình xác suất thường nhỉnh hơn từng mô hình riêng.

In [ ]:
def run_xgb():
    oof = np.zeros(len(y_tr)); pred = np.zeros(len(y_te))
    for a, b in folds:
        m = xgb.XGBClassifier(n_estimators=5000, learning_rate=0.05, max_depth=6, subsample=0.8, colsample_bytree=0.8, tree_method='hist',
                              enable_categorical=True, eval_metric='auc', early_stopping_rounds=100, random_state=SEED)
        m.fit(X_tr.iloc[a], y_tr[a], eval_set=[(X_tr.iloc[b], y_tr[b])], verbose=False)
        oof[b] = m.predict_proba(X_tr.iloc[b])[:, 1]; pred += m.predict_proba(X_te)[:, 1] / len(folds)
    return oof, pred

def run_cat():
    Xc_tr, Xc_te = X_tr.copy(), X_te.copy()
    for c in cat_cols:
        Xc_tr[c] = Xc_tr[c].astype(str); Xc_te[c] = Xc_te[c].astype(str)
    oof = np.zeros(len(y_tr)); pred = np.zeros(len(y_te))
    for a, b in folds:
        m = catboost.CatBoostClassifier(iterations=3000, learning_rate=0.08, depth=6, eval_metric='AUC', cat_features=cat_cols,
                                        random_seed=SEED, verbose=False, early_stopping_rounds=100)
        m.fit(Xc_tr.iloc[a], y_tr[a], eval_set=(Xc_tr.iloc[b], y_tr[b]))
        oof[b] = m.predict_proba(Xc_tr.iloc[b])[:, 1]; pred += m.predict_proba(Xc_te)[:, 1] / len(folds)
    return oof, pred

t = time.time(); oof_xgb, pred_xgb = run_xgb(); t_xgb = time.time() - t
t = time.time(); oof_cat, pred_cat = run_cat(); t_cat = time.time() - t
oof_avg, pred_avg = (oof_lgb + oof_xgb + oof_cat) / 3, (pred_lgb + pred_xgb + pred_cat) / 3
res = pd.DataFrame({'mô hình': ['logistic regression', 'LightGBM', 'XGBoost', 'CatBoost', 'trung bình ba mô hình'],
                    'OOF AUC': [auc_lr.mean(), auc_oof_lgb, roc_auc_score(y_tr, oof_xgb), roc_auc_score(y_tr, oof_cat), roc_auc_score(y_tr, oof_avg)],
                    'test AUC': [np.nan, auc_te_lgb, roc_auc_score(y_te, pred_xgb), roc_auc_score(y_te, pred_cat), roc_auc_score(y_te, pred_avg)],
                    'giây': [np.nan, t_lgb, t_xgb, t_cat, np.nan]})
table(res, 'Năm cách, cùng khuôn 5 fold', fmt={'OOF AUC': '{:.4f}', 'test AUC': '{:.4f}', 'giây': '{:.0f}'}, best='OOF AUC')

fig, ax = plt.subplots(figsize=(9, 3.4))
ax.barh(res['mô hình'], res['OOF AUC'], color=[C['gray'], C['blue'], C['blue'], C['blue'], C['green']]); ax.invert_yaxis()
lo = res['OOF AUC'].min() - 0.01; ax.set_xlim(lo, res['OOF AUC'].max() + 0.006)
ax.set(xlabel='OOF AUC', title='Ba thư viện boosting gần như ngang nhau, cùng vượt logistic'); ax.grid(axis='y', visible=False); bar_labels(ax, '{:.4f}')
plt.tight_layout(); plt.show()
check('trung bình ba mô hình không thua mô hình đơn tốt nhất quá 0.001', res['OOF AUC'].iloc[4] > res['OOF AUC'].iloc[1:4].max() - 0.001)
note(f"Trung bình ba mô hình: OOF <b>{res['OOF AUC'].iloc[4]:.4f}</b> so với mô hình đơn tốt nhất {res['OOF AUC'].iloc[1:4].max():.4f}. "
     f"Kiểm bằng OOF trước rồi mới tin, đúng cách làm của bài 8.4.")

## 8. Tóm tắt

In [ ]:
summary = [('logistic regression, OOF AUC', f'{auc_lr.mean():.4f}'),
           ('LightGBM, OOF / test AUC', f'{auc_oof_lgb:.4f} / {auc_te_lgb:.4f}'),
           ('số cây khi dừng: learning rate 0.3 / 0.03', f'{lr_df.iloc[0, 1]} / {lr_df.iloc[2, 1]}'),
           (f'bỏ cột {top} / bỏ cột {second}', f"{ab['đổi'].iloc[1]:+.4f} / {ab['đổi'].iloc[2]:+.4f} AUC"),
           ('trung bình ba mô hình, OOF / test AUC', f"{res['OOF AUC'].iloc[4]:.4f} / {res['test AUC'].iloc[4]:.4f}")]
table(pd.DataFrame(summary, columns=['điều đo', 'kết quả']), 'Các con số chính của notebook')

**Thử thêm khi dạy:** đổi `num_leaves=31` thành 7 hay 127 rồi xem số cây khi dừng; bỏ early stopping (đặt `rounds=300`) để thấy AUC fold có thể đi xuống; bỏ cột `relationship` cùng lúc với cột mạnh nhất.